Amazon Electronics Data Preprocessing

Load, explore, clean, and prepare the Amazon Electronics Reviews dataset for sentiment classification and review summarization.

In [1]:
# import the necessary libraries 
# Data manipulation
import pandas as pd
import numpy as np

# Working with file paths
from pathlib import Path

# Dataset splitting
from sklearn.model_selection import train_test_split

Instead of loading the full datasets we are starting with previewing 10 rows from each file. Reason being that the reviews dataset contains over 18 million rows which is a lot for the lab to process and load. It will significantly slow the notebook and take up too much storage. Setting the path for the review and metadata path and then reviewing the structure by looking attributes. 

In [2]:
# Set the file paths for both the review and metadata jsonl files just for a preview of 10 rows 
reviews_path = Path("../../Electronics.jsonl.gz")
metadata_path = Path("../../meta_Electronics.jsonl.gz")

In [3]:
# Make sure that they exist 
print("Reviews file exists:", reviews_path.exists())
print("Metadata file exists:", metadata_path.exists())

Reviews file exists: True
Metadata file exists: True


In [4]:
# Before I try to load the entire dataset which consists of millions of rows I just want to see a sample of the reviews data (10 rows)
reviews_preview = pd.read_json(
    reviews_path,
    lines=True,
    nrows=10
)

reviews_preview

# Looking at this dataset the important columns would be rating, title, text, and asin or parent_asin to join to the metadata dataset to know the product group.

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,3,Smells like gasoline! Going back!,First & most offensive: they reek of gasoline ...,[{'small_image_url': 'https://m.media-amazon.c...,B083NRGZMM,B083NRGZMM,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,2022-07-18 22:58:37.948,0,True
1,1,Didn’t work at all lenses loose/broken.,These didn’t work. Idk if they were damaged in...,[],B07N69T6TM,B07N69T6TM,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,2020-06-20 18:42:29.731,0,True
2,5,Excellent!,I love these. They even come with a carry case...,[],B01G8JO5F2,B01G8JO5F2,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,2018-04-07 09:23:37.534,0,True
3,5,Great laptop backpack!,I was searching for a sturdy backpack for scho...,[],B001OC5JKY,B001OC5JKY,AGGZ357AO26RQZVRLGU4D4N52DZQ,2010-11-20 18:41:35.000,18,True
4,5,Best Headphones in the Fifties price range!,I've bought these headphones three times becau...,[],B013J7WUGC,B07CJYMRWM,AG2L7H23R5LLKDKLBEF2Q3L2MVDA,2023-02-17 02:39:41.238,0,True
5,5,Great Fan! I’m a FAN!,"Light weight, quiet and totally awesome!!! It ...",[],B072DSHKCH,B07CML419K,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,2021-11-21 19:28:01.041,0,True
6,5,solid sound for the price,Update 2-they sent a new warranty replacement....,[],B07BHHB5RH,B07BHHB5RH,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,2019-08-06 22:34:39.386,0,True
7,5,Love the headphones - great range,These are fantastic headphones and I love that...,[],B07BND376H,B09S6Y5BRG,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,2018-11-04 18:40:31.659,9,True
8,5,Five Stars,pretty good for the price.,[],B002HWRZ2K,B01LW71IBJ,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,2016-02-29 19:02:51.000,0,True
9,5,BUY THIS THANG,yes.. so good. just buy it. my favorite featu...,[],B00WK47VEW,B017T99JPG,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,2016-02-29 18:59:25.000,0,True


In [5]:
# Next will look at what information is provided for each product 
metadata_preview = pd.read_json(
    metadata_path,
    lines=True,
    nrows=10
)

metadata_preview

# This set is a bit interesting because there is a average rating and rating_number which may be useful in the analysis or sentiment classification. 
# The main fields we may need are main_category, title, average_rating, rating, parent_asin to join and maybe store for EDA

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together
0,All Electronics,FS-1051 FATSHARK TELEPORTER V3 HEADSET,3.5,6,[],[Teleporter V3 The “Teleporter V3” kit sets a ...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Fat Shark,"[Electronics, Television & Video, Video Glasses]","{'Date First Available': 'August 2, 2014', 'Ma...",B00MCW7G9M,NaN
1,All Electronics,Ce-H22B12-S1 4Kx2K Hdmi 4Port,5.0,1,"[UPC: 662774021904, Weight: 0.600 lbs]",[HDMI In - HDMI Out],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],SIIG,"[Electronics, Television & Video, Accessories,...",{'Product Dimensions': '0.83 x 4.17 x 2.05 inc...,B00YT6XQSE,NaN
2,Computers,Digi-Tatoo Decal Skin Compatible With MacBook ...,4.5,246,[WARNING: Please IDENTIFY MODEL NUMBER on the ...,[],19.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'AL 2Sides Video', 'url': 'https://...",Digi-Tatoo,"[Electronics, Computers & Accessories, Laptop ...","{'Brand': 'Digi-Tatoo', 'Color': 'Fresh Marble...",B07SM135LS,NaN
3,AMAZON FASHION,NotoCity Compatible with Vivoactive 4 band 22m...,4.5,233,[☛NotoCity 22mm band is designed for Vivoactiv...,[],9.99,[{'thumb': 'https://m.media-amazon.com/images/...,[],NotoCity,"[Electronics, Wearable Technology, Clips, Arm ...","{'Date First Available': 'May 29, 2020', 'Manu...",B089CNGZCW,NaN
4,Cell Phones & Accessories,Motorola Droid X Essentials Combo Pack,3.8,64,"[New Droid X Essentials Combo Pack, Exclusive ...",[all Genuine High Quality Motorola Made Access...,14.99,[{'thumb': 'https://m.media-amazon.com/images/...,[],Verizon,"[Electronics, Computers & Accessories, Compute...",{'Product Dimensions': '11.6 x 6.9 x 3.1 inche...,B004E2Z88O,NaN
5,Sports & Outdoors,Raymarine Wi-Fish DownVision Blackbox Sonar wi...,3.5,25,[Black box Wi-Fi CHIRP DownVision sonar module...,[Transform your smartphone into a powerful CHI...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Raymarine,"[Electronics, Car & Vehicle Electronics, Marin...",{'Item Package Dimensions L x W x H': '9.3 x 8...,B00TX536EK,NaN
6,Cell Phones & Accessories,"QGHXO Band for Garmin Vivofit 4, Soft Silicone...",4.4,707,[Personalized Your Garmin Vivofit 4 Activity T...,"[Compatibility, Custom designed for your preci...",14.89,[{'thumb': 'https://m.media-amazon.com/images/...,[],QGHXO,"[Electronics, Wearable Technology, Arm & Wrist...",{'Package Dimensions': '6.85 x 4.37 x 1.1 inch...,B07BJ7ZZL7,NaN
7,Industrial & Scientific,Protech iPhone 4 microscope 60X Lens with illu...,3.8,6,[iPhone 4 Microscope with White 2-LED and Dete...,[This accessory will convert your iPhone 4 int...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],ProTech,"[Electronics, Camera & Photo, Binoculars & Sco...","{'Light Source Type': 'LED', 'Color': 'Blue,Wh...",B005G99O2U,NaN
8,Computers,MOSISO Plastic Hard Shell Case & Keyboard Cove...,5.0,2,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],MOSISO,"[Electronics, Headphones, Earbuds & Accessorie...","{'Brand': 'MOSISO', 'Item Weight': '8 ounces',...",B01MCZP7RF,NaN
9,NaN,"Fishfinder, Depth Finder Poly Sun Cover for 3""...",4.4,86,"[Made of poly material, Available in black, Dr...",[Affordable and convenient way to cover your s...,10.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Unboxing Gamin Vivid ', 'url': 'ht...",Westlake Market,"[Electronics, Car & Vehicle Electronics, Marin...",{'Item Package Dimensions L x W x H': '7.32 x ...,B00R6R82HS,NaN


In [6]:
# Review the datatype for each column 
reviews_preview.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   rating             10 non-null     int64         
 1   title              10 non-null     str           
 2   text               10 non-null     str           
 3   images             10 non-null     object        
 4   asin               10 non-null     str           
 5   parent_asin        10 non-null     str           
 6   user_id            10 non-null     str           
 7   timestamp          10 non-null     datetime64[ms]
 8   helpful_vote       10 non-null     int64         
 9   verified_purchase  10 non-null     bool          
dtypes: bool(1), datetime64[ms](1), int64(2), object(1), str(5)
memory usage: 862.0+ bytes


In [7]:
# Review the data type for each column 
metadata_preview.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   main_category    9 non-null      str    
 1   title            10 non-null     str    
 2   average_rating   10 non-null     float64
 3   rating_number    10 non-null     int64  
 4   features         10 non-null     object 
 5   description      10 non-null     object 
 6   price            5 non-null      float64
 7   images           10 non-null     object 
 8   videos           10 non-null     object 
 9   store            10 non-null     str    
 10  categories       10 non-null     object 
 11  details          10 non-null     object 
 12  parent_asin      10 non-null     str    
 13  bought_together  0 non-null      float64
dtypes: float64(3), int64(1), object(6), str(4)
memory usage: 1.2+ KB


Now that we have reviewed the columns, I am going to take a random sample of the metadata dataset to then clean, transform, and join with the review dataset. 

In [8]:
# Display the file size in GB
print(f"{metadata_path.stat().st_size / (1024**3):.2f} GB")

1.22 GB


In [9]:
# Load the metadata dataset in chunks
metadata_chunks = []

for chunk in pd.read_json(
    metadata_path,
    lines=True,
    chunksize=50000
):
    metadata_chunks.append(chunk)

In [10]:
# Combine all metadata chunks
metadata_df = pd.concat(metadata_chunks, ignore_index=True)

In [11]:
# Check the size of the metadata dataset
metadata_df.shape

(1610012, 16)

In [12]:
# Check for duplicate products
metadata_df["parent_asin"].duplicated().sum()

np.int64(0)

In [13]:
# Keep only the columns needed for the project
metadata_df = metadata_df[
    [
        "parent_asin",
        "main_category",
        "title",
        "average_rating",
        "rating_number"
    ]
]

In [14]:
# Randomly sample 10,000 products
metadata_df = metadata_df.sample(
    n=10000,
    random_state=42
).reset_index(drop=True)

In [15]:
metadata_df.shape

(10000, 5)

In [17]:
# Get the sampled product IDs
product_ids = set(metadata_df["parent_asin"])

In [18]:
# Load reviews for the sampled products
review_chunks = []

for chunk in pd.read_json(
    reviews_path,
    lines=True,
    chunksize=50000
):

    filtered_chunk = chunk[
        chunk["parent_asin"].isin(product_ids)
    ]

    review_chunks.append(filtered_chunk)

reviews_df = pd.concat(review_chunks, ignore_index=True)

In [19]:
# Check the size of the review dataset
reviews_df.shape

(280022, 10)

In [20]:
reviews_df.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,4,"Works as designed, but not easy to install","To be honest, I had a lot of difficulty gettin...",[],B007Q8GOGI,B007Q8GOGI,AFSKPY37N3C43SOI5IEXEK5JSIYA,2013-04-13 12:35:22.000,0,False
1,1,cheap plastic,i was expecting for this price at least someth...,[],B07H9DWMT2,B07H9FG9SD,AHGAOIZVODNHYMNCBV4DECZH42UQ,2019-11-12 20:21:33.415,1,True
2,5,What a beautiful TV. The price was fair,What a beautiful TV. The price was fair. I o...,[],B0036WT3ZM,B0036WT3ZM,AHZ6XMOLEWA67S3TX7IWEXXGWSOA,2011-05-25 19:14:54.000,0,True
3,5,cord tamer stick on clips,Purchased these to tame a few cords... I place...,[],B07FTCRKB7,B07FTCRKB7,AFZUK3MTBIBEDQOPAK3OATUOUKLA,2019-10-15 18:39:06.815,2,True
4,4,worked well,My husband tried these for work. He says that...,[],B004O292WC,B004O292WC,AFZUK3MTBIBEDQOPAK3OATUOUKLA,2014-09-17 21:00:59.000,1,True


In [21]:
# Remove unnecessary columns
reviews_df = reviews_df.drop(
    columns=["images", "verified_purchase", "user_id"]
)

In [22]:
# Display the data types of each column
# All the data types look good but we should check for whitespaces
reviews_df.dtypes

rating                   int64
title                      str
text                       str
asin                       str
parent_asin                str
timestamp       datetime64[ms]
helpful_vote             int64
dtype: object

In [23]:
# Check for duplicate reviews
reviews_df.duplicated().sum()

np.int64(3020)

In [24]:
# Remove duplicate reviews
reviews_df = reviews_df.drop_duplicates()

In [25]:
# Verify that duplicate reviews were removed
reviews_df.duplicated().sum()

np.int64(0)

In [26]:
# Remove extra spaces at the beginning and end of the review title and text
reviews_df["title"] = reviews_df["title"].str.strip()
reviews_df["text"] = reviews_df["text"].str.strip()

In [27]:
# Replace multiple spaces with a single space
reviews_df["title"] = reviews_df["title"].str.replace(r"\s+", " ", regex=True)
reviews_df["text"] = reviews_df["text"].str.replace(r"\s+", " ", regex=True)

In [28]:
# Count reviews with empty text
(reviews_df["text"] == "").sum()

np.int64(252)

In [29]:
# Display reviews with empty review text
reviews_df[reviews_df["text"] == ""]

,rating,title,text,asin,parent_asin,timestamp,helpful_vote
2490,5,GREAT quality!,,B095HFV9DC,B095HF8Z2L,2022-12-26 01:12:51.833,0
2504,5,So far so good,,B08SB72Q8F,B08SB72Q8F,2022-07-20 02:58:34.435,0
4247,5,Works perfectly with my Ambient Weather station,,B09CGY1SWR,B09LHDS2NG,2022-03-22 02:07:31.812,12
6109,5,Easy installation,,B08QJ1ZK9S,B08QJ1ZK9S,2022-09-01 17:07:21.526,1
8238,5,Love it!,,B084DCJKSL,B08YGKCHNF,2022-01-04 19:11:31.048,0
...,...,...,...,...,...,...,...
279111,5,Great quality for the price.,,B09V1F24X4,B09V1F24X4,2022-03-24 22:40:30.412,1
279473,5,Great product!,,B094XY5XLP,B0B9NVVK3B,2022-12-28 04:17:23.401,0
279485,5,A,,B005EJH6RW,B07538T649,2023-07-08 16:25:41.597,0
279589,5,scratches easy but gets the job done,,B08GSG794B,B08GSG794B,2022-08-22 00:58:27.768,0


In [30]:
# Remove reviews with empty review text because that will not help with the summary generation piece
reviews_df = reviews_df[reviews_df["text"] != ""]

In [31]:
print(reviews_df.shape)

(276750, 7)


In [32]:
# Merge the reviews with the metadata
merged_df = pd.merge(
    reviews_df,
    metadata_df,
    on="parent_asin",
    how="left"
)

In [33]:
# Rename the title columns
merged_df = merged_df.rename(columns={
    "title_x": "review_title",
    "title_y": "product_title"
})

In [35]:
# Rearrange the columns
merged_df = merged_df[
    [
        "parent_asin",
        "product_title",
        "main_category",
        "average_rating",
        "rating_number",
        "asin",
        "rating",
        "review_title",
        "text",
        "timestamp",
        "helpful_vote"
    ]
]

In [36]:
# Display the size of the final dataset
merged_df.shape

(276750, 11)

In [37]:
# Display the first five rows
merged_df.head()

,parent_asin,product_title,main_category,average_rating,rating_number,asin,rating,review_title,text,timestamp,helpful_vote
0,B007Q8GOGI,3M Natural View Fingerprint Fading Screen Prot...,Computers,3.3,41,B007Q8GOGI,4,"Works as designed, but not easy to install","To be honest, I had a lot of difficulty gettin...",2013-04-13 12:35:22.000,0
1,B07H9FG9SD,Mount Genie Pedestal for Nest Mini (2nd Gen) a...,All Electronics,4.6,4023,B07H9DWMT2,1,cheap plastic,i was expecting for this price at least someth...,2019-11-12 20:21:33.415,1
2,B0036WT3ZM,Samsung PN58C550 58-Inch 1080p Plasma HDTV (Bl...,All Electronics,3.4,85,B0036WT3ZM,5,What a beautiful TV. The price was fair,What a beautiful TV. The price was fair. I ori...,2011-05-25 19:14:54.000,0
3,B07FTCRKB7,"30 Pack Self-Adhesive Cable Clips, Nylon Cable...",Tools & Home Improvement,4.0,46,B07FTCRKB7,5,cord tamer stick on clips,Purchased these to tame a few cords... I place...,2019-10-15 18:39:06.815,2
4,B004O292WC,Vertex VX-231-G7UN Business/Industrial Portabl...,Cell Phones & Accessories,3.9,8,B004O292WC,4,worked well,My husband tried these for work. He says that ...,2014-09-17 21:00:59.000,1


In [38]:
# Count the number of unique products
merged_df["parent_asin"].nunique()

9992

In [39]:
# Count the number of reviews for each product
reviews_per_product = (
    merged_df.groupby("parent_asin")
    .size()
    .reset_index(name="number_of_reviews")
)

reviews_per_product.head()

,parent_asin,number_of_reviews
0,0375505458,7
1,0446673145,7
2,059454582X,11
3,0594626757,6
4,1412789788,3


In [40]:
# Display the products with the most reviews
reviews_per_product.sort_values(
    by="number_of_reviews",
    ascending=False
).head(10)

,parent_asin,number_of_reviews
8412,B08YGKCHNF,7386
8299,B08TMJFSV3,7351
4861,B071J3PFJJ,5579
2760,B00K8GZED4,5398
8711,B0995JS7K9,5396
4546,B01MTY7MSK,4789
9802,B0BTT1Z16D,4661
2578,B00HQWLHJI,4142
4462,B01M3575QC,3704
9644,B0BHQ3NSQ3,3686


In [41]:
# Display the products with the fewest reviews
reviews_per_product.sort_values(
    by="number_of_reviews",
    ascending=True
).head(10)

,parent_asin,number_of_reviews
6961,B07X8MYSQW,1
2360,B00ET3W372,1
7423,B0863FMDPB,1
4101,B01EUIKX04,1
4100,B01ETAKVXW,1
8928,B09JC17HZF,1
5987,B07HRW4MQN,1
5986,B07HRD91R6,1
4099,B01ESQCRGQ,1
2368,B00EYY5ZOO,1


In [ ]:
# Save the processed dataset as a compressed CSV
merged_df.to_csv(
    "processed_reviews.csv.gz",
    index=False,
    compression="gzip"
)